# Q-learning training and evaluation
#### Payload type: Pure CPU workload + nested Python loops

In [ ]:
import numpy as np
import pygame
import gymnasium as gym
from gymnasium import spaces
import mediapy as media
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

In [ ]:
# We here code the simple MDP. When reading the code you can skip the rendering part.
# Adapted from https://gymnasium.farama.org/tutorials/gymnasium_basics/environment_creation/
class GridWorldEnv(gym.Env):
    
    action_to_direction = {
            0: np.array([1, 0]),
            1: np.array([0, 1]),
            2: np.array([-1, 0]),
            3: np.array([0, -1]),
        }

    def __init__(self):
        self.size = 4  # The size of the square grid
        self.window_size = 256  # The size of the PyGame window

        # Observations are dictionaries with the agent's and the target's location.
        # Each location is encoded as an element of {0, ..., `size`}^2, i.e. MultiDiscrete([size, size]).
        self.observation_space = spaces.Dict(
            {
                "agent": spaces.Box(0, self.size - 1, shape=(2,), dtype=int),
                "target": spaces.Box(0, self.size - 1, shape=(2,), dtype=int),
            }
        )

        # We have 4 actions, corresponding to "right", "up", "left", "down"
        self.action_space = spaces.Discrete(4)

    def _get_obs(self):
        return {"agent": self._agent_location, "target": self._target_location}

    def _get_info(self):
        return {
            "distance": np.linalg.norm(
                self._agent_location - self._target_location, ord=1
            )
        }

    def reset(self, seed=None, target=None):
        super().reset(seed=seed)

        if target is None:
            # Choose the target's location uniformly at random
            self._target_location = self.np_random.integers(0, self.size, size=2, dtype=int)
        else:
            assert len(target) == 2
            self._target_location = np.array(target)

        # We will sample the agent's location randomly until it does not coincide with the target's location
        self._agent_location = self._target_location
        while np.array_equal(self._agent_location, self._target_location):
            self._agent_location = self.np_random.integers(
                0, self.size, size=2, dtype=int
            )

        observation = self._get_obs()
        info = self._get_info()

        return observation, info

    def step(self, action):
        # Map the action (element of {0,1,2,3}) to the direction we walk in
        direction = self.action_to_direction[action]
        # We use `np.clip` to make sure we don't leave the grid
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )
        # An episode is done if the agent has reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)
        reward = 1 if terminated else 0  # Binary sparse rewards
        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, info

    def render(self, action=None):
        return self._render_frame(action)

    def _render_frame(self, action):
        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_square_size = (
            self.window_size / self.size
        )  # The size of a single grid square in pixels

        # First we draw the target
        pygame.draw.rect(
            canvas,
            (255, 0, 0),
            pygame.Rect(
                pix_square_size * self._target_location,
                (pix_square_size, pix_square_size),
            ),
        )
        # Now we draw the agent
        pygame.draw.circle(
            canvas,
            (0, 0, 255),
            (self._agent_location + 0.5) * pix_square_size,
            pix_square_size / 3,
        )
        if action is not None:
            pygame.draw.line(
                canvas,
                1,
                (self._agent_location + 0.5) * pix_square_size,
                (self._agent_location + 0.5 + self.action_to_direction[action]/3) * pix_square_size,
                width=2
            )

        # Finally, add some gridlines
        for x in range(self.size + 1):
            pygame.draw.line(
                canvas,
                0,
                (0, pix_square_size * x),
                (self.window_size, pix_square_size * x),
                width=3,
            )
            pygame.draw.line(
                canvas,
                0,
                (pix_square_size * x, 0),
                (pix_square_size * x, self.window_size),
                width=3,
            )

        return np.transpose(
            np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2)
        )

In [ ]:
# Since we know the full MDP, we can use dynamic programming (fixed-point algorithm) to get the value function.
# Note that we could also use other methods like solving a set of linear equations.
policy = lambda s: [0.25,0.25,0.25,0.25] # our random policy, it does not depend on the state
gamma = 0.9 # discount factor
target = (3,3)
directions = np.array([[1,0],[0,1],[-1,0],[0,-1]])
V_star = np.zeros((4,4)) # (col, row), starting top left
Q_star = np.zeros((4,4,4))
for steps in range(200):
    # Iteration over states
    for state_idx in range(16):
        state = (state_idx%4,int(state_idx/4)) 
        if state==target:
            continue
        action_probs = policy(state)
        assert sum(action_probs) == 1.
        subseq_states = np.clip(state+directions,0,3)
        V_subseq_states = np.array([V_star[tuple(idx)] for idx in subseq_states])
        rewards = (subseq_states==target).all(1).astype('float')
        V_star[state] = np.max(rewards+gamma*V_subseq_states)
        Q_star[state] = rewards + gamma * V_subseq_states

print(f'Optimal state-values:\n{np.round(V_star.T,3)}')
print(f'Optimal state-action-value function for going right:\n{np.round(Q_star[:,:,0].T,3)}')
dir_str = lambda idxs: np.array(list(map(lambda i: ['right','down','left','right'][i], idxs)))
print(f'The optimal policy (there exist multiple optimal policies):\
      \n{np.array(dir_str(np.argmax(Q_star,axis=2).T.reshape(-1))).reshape(4,4)}')

In [ ]:
## Solution
# policy, based on Q
def epsilon_greedy_Q_policy(s, Q, eps=0.1):
    if np.random.uniform(0, 1) <= eps:
        return np.random.choice(np.arange(4))
    # Random sample from all equally optimal options
    return np.random.choice(np.where(Q[tuple(s)]==np.max(Q[tuple(s)]))[0])
# parameters
total_episodes = 20000
max_steps_per_episode = 100
gamma = 0.9
alpha = 0.2
target = (3,3)
# Initialize Q.
Q_pi_hat = np.zeros((4, 4, 4))
# Lists to keep track of training statistics.
episode_lengths = []
max_errors = []
avg_errors = []
env = GridWorldEnv()
for episode in range(total_episodes):
    s = env.reset(target=target)[0]['agent']
    step = 0
    # Generate a trajectory for a limited number of steps.
    while step<max_steps_per_episode:
        step += 1
        a = epsilon_greedy_Q_policy(s, Q_pi_hat)  # Pick action.
        next_s, r, terminal, _ = env.step(a)  # Take a step in the environment.
        next_s = next_s["agent"]
        delta = r + gamma * max(Q_pi_hat[tuple(next_s)]) - Q_pi_hat[tuple(s)][a]
        Q_pi_hat[tuple(s)][a] += alpha * delta  # Q-learning update.
        s = next_s
        if terminal:
            break
    ## metrics for plotting
    episode_lengths.append(float(step))
    max_errors.append(np.max(np.abs(Q_star - Q_pi_hat)))
    avg_errors.append(np.mean(np.abs(Q_star - Q_pi_hat)))

# plotting
fig, [ax1,ax2] = plt.subplots(2,1)
ax1.plot(avg_errors,label='avg. Q error')
ax1.plot(max_errors, label='max. Q error')
ax1.axhline(0,c='grey',linestyle='--')
ax1.set_ylabel('error')
ax1.legend()
ax2.plot(gaussian_filter(episode_lengths,100))
opt_avg_eps_length = 1/15*(2*1+3*2+4*3+3*4+2*5+1*6)
ax2.axhline(opt_avg_eps_length, c='grey', linestyle='--')
ax2.set_xlabel('episode')
ax2.set_ylabel('avg. episode length')
        
# Trial runs with greedy policy
total_test_episodes=10000
length_test_episodes=[]
for episode in range(total_test_episodes):
    s = env.reset(target=target)[0]['agent']
    step = 0
    while step<max_steps_per_episode:
        step += 1
        a = epsilon_greedy_Q_policy(s, Q_pi_hat, eps=0.)
        next_s, r, terminal, _ = env.step(a) 
        next_s = next_s['agent']
        s = next_s
        if terminal:
            break
    length_test_episodes.append(step)
print(f'Mean episode length of our policy: {np.mean(length_test_episodes)}, of the optimal policy: {opt_avg_eps_length}')
